# Podcast Emotion and Content Analysis

This project explores the linguistic, emotional, and topical characteristics of podcast episodes using Natural Language Processing (NLP) methods.  
By combining pre-trained language models and structured metadata, we aim to uncover the emotional tone, thematic categories, and audience alignment of each podcast episode.

Our pipeline operates at two complementary levels:
- **Turn-level analysis:** Emotion detection and safety tagging for individual speaker turns.
- **Episode-level analysis:** Aggregation of features to classify episodes into genres, estimate audience interests, and identify potentially unsafe content.


### Motivation

Podcasts are a rich and fast-growing medium, with content ranging from personal storytelling to political commentary.  
However, the scale and diversity of podcast data make manual analysis impractical.  

Automatic understanding of podcast episodes can support:
- **Listeners**, by improving recommendation and discovery systems.
- **Producers**, by providing insights into emotional tone and audience engagement.
- **Platforms**, by enabling automated content moderation and brand safety filtering.

Despite advances in text-based NLP, podcasts present unique challenges: conversational structure, speaker variability, and diverse topics.  
This project seeks to address these challenges by building an interpretable and scalable NLP pipeline tailored for podcast data.


### Project Goals

The primary objectives of this project are:

1. **Turn-level emotion detection** — Identify emotions (e.g., joy, anger, sadness) in each speaker turn using a pre-trained emotion recognition model.
2. **Episode-level categorization** — Classify each episode into broad categories such as *Technology*, *Comedy*, or *Politics*.
3. **Brand safety detection** — Flag episodes that contain unsafe or sensitive content, including hate speech, violence, or adult themes.
4. **Audience alignment prediction** — Estimate the likely audience or interest group best aligned with each episode based on its emotional and linguistic profile.

Collectively, these tasks aim to produce an interpretable summary of each episode — showing how speakers’ emotions, topics, and tone interact across the conversation.


### Expected Outcomes

By the end of this project, we aim to:
- Deliver a fully documented pipeline capable of processing large-scale podcast transcript data.
- Demonstrate emotion and safety predictions on speaker turns using transformer-based models.
- Aggregate turn-level features to classify and describe entire episodes.
- Provide interpretable summaries for each episode showing emotion distribution, category prediction, and safety assessment.

This proof of concept validates that pre-trained NLP models can be combined and scaled to perform fine-grained, multi-level podcast analysis.


## 1. Data Loading and Overview

To start, we'll load the sample datasets, `episodeLevelDataSample` and `speakerTurnDataSample`. The full datasets are over 20 GB, so we're using these smaller versions for the demonstration in this milestone.

In [ ]:
import gzip
import json
import pandas as pd

episode_level = []
with gzip.open("data/episodeLevelDataSample.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        episode_level.append(json.loads(line))

episode_df = pd.DataFrame(episode_level)

In [ ]:
speakerTurn = []
with gzip.open("data/speakerTurnDataSample.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        speakerTurn.append(json.loads(line))

speaker_df = pd.DataFrame(speakerTurn)

### Columns and Examples

#### Episode Level

The `episodeLevelDataSample` dataset provides metadata for each podcast episode in the sample, including information like the episode title, description, podcast title, and categories. The mp3url in this dataset is particularly important as it acts as a unique identifier for each episode, allowing us to link it with the corresponding speaker turn data later on. We'll use this data to understand the context of the speaker turns and potentially for tasks like episode classification and analysis based on episode-level features.

In [ ]:
print("episodeLevelDataSample columns: ", episode_df.keys())

print("\n episodeLevelDataSample examples: ")
episode_df.sample(5) # Display 5 random samples from the dataset

#### Speaker Turn Level

The `speakerTurnDataSample` dataset contains information about individual speaker turns within each podcast episode. This includes the most important one which is the actual text spoken (`turnText`), the identified speaker, the start and end times of the turn, and the mp3url to link it back to the episode metadata in the episodeLevelDataSample.

In [ ]:
print("speakerTurnDataSample columns: ", speaker_df.keys())

print("\n speakerTurnDataSample examples: ")
speaker_df.sample(5) # Display 5 random samples from the dataset

Given the large number of columns in these datasets, we want to select a subset of columns from each that are relevant to our analysis and discard the rest.

## 2. Data Preprocessing

In this section, we perform a series of preprocessing steps to prepare the dataset for downstream analysis and modeling.

The primary objective is to retain only the most relevant information from the raw `speakerTurnData` and `episodeLevelData` sources while ensuring that textual fields — particularly `turnText` — are clean, standardized, and ready for the tasks.

This involves:

* Selecting key columns that capture speaker and episode metadata.
* Cleaning and normalizing text data to remove noise (e.g., music markers, punctuation, excess whitespace).
* Filtering out irrelevant or empty content.
* Aggregating speaker turns and merging with episode-level information for a unified dataset.

### 2.1 Select Relevant Columns (Speaker-Turn Data)
We extract a subset of columns from the original dataset to create a compact version containing the most informative features for each speaker turn.

| Column                  | Description                                                |
| :---------------------- | :--------------------------------------------------------- |
| **turnText**            | Transcribed text of the speaker’s turn                     |
| **speaker**             | Identifier for the speaker                                 |
| **mp3url**              | Link to the episode (useful for grouping turns by episode) |
| **turnCount**           | Sequential position of the turn within an episode          |
| **inferredSpeakerRole** | Predicted speaker role (e.g., host, guest)                 |
| **startTime**           | Start time of a turn                                       |
| **endTime**             | End time of a turn                                         |


In [ ]:
cols = ['turnText', 'speaker', 'mp3url', 'turnCount', 'inferredSpeakerRole',  'startTime', 'endTime']
speaker_df1 = pd.DataFrame(speakerTurn)[cols].copy()

### 2.2 Text Normalization

We standardize the text to lowercase and normalize whitespace.
This step ensures consistent tokenization and reduces vocabulary redundancy (e.g., “The” and “the” are treated the same).

In [ ]:
speaker_df1['turnText'] = (
    speaker_df1['turnText']
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

### 2.3 Noise Removal

The transcripts contain several types of non-linguistic artifacts such as musical cues, blank-audio markers, ellipses, and transcription symbols.

These tokens provide no semantic value and can bias subsequent NLP analysis.

To obtain cleaner text representations, we remove all non-speech elements and normalize leftover formatting.

In [ ]:
import re

# --- 1. Define stronger unwanted patterns ---
# Matches variations like:
# [music], (music playing), [background music fades], music:, blank_audio etc.
unwanted_patterns = [
    r'\[.*?music.*?\]',       # [music], [background music], [music playing]
    r'\(.*?music.*?\)',       # (music), (music playing), (background music)
    r'\[.*?blank_audio.*?\]', # [blank_audio] or variations
    r'\(.*?blank_audio.*?\)', # (blank_audio)
    r'\bmusic\b',             # standalone 'music'
    r'\bblank_audio\b'        # standalone 'blank_audio'
]

# --- 2. Apply removal ---
for pattern in unwanted_patterns:
    speaker_df1['turnText'] = speaker_df1['turnText'].str.replace(pattern, '', regex=True)

# --- 3. Clean malformed or stray brackets ---
def clean_brackets(text):
    """Removes stray [ or ] while preserving valid bracketed expressions."""
    return re.sub(r'(\[[^\[\]]+\])|[\[\]]', lambda m: m.group(1) or '', text)

speaker_df1['turnText'] = speaker_df1['turnText'].apply(clean_brackets)

# --- 4. Remove conversation markers or ellipses ---
speaker_df1['turnText'] = (
    speaker_df1['turnText']
    .str.replace('>>', '', regex=False)
    .str.replace(r'\.\.\.', '', regex=True)  # remove ...
)

# --- 5. Remove near-empty or symbol-only entries ---
speaker_df1 = speaker_df1[speaker_df1['turnText'].str.len() > 1]
speaker_df1 = speaker_df1[~speaker_df1['turnText'].str.match(r'^\W*$')]

# --- 6. Final whitespace normalization ---
speaker_df1['turnText'] = speaker_df1['turnText'].str.replace(r'\s+', ' ', regex=True).str.strip()


### 2.4 Example Output

In [ ]:
target_url = "https://traffic.megaphone.fm/APO4719798278.mp3"
speaker_df_example = speaker_df1[speaker_df1['mp3url'] == target_url]
speaker_df_example[['speaker','turnText']]

----------- 

# TO BE REMOVED

In [ ]:
target_url = "https://traffic.megaphone.fm/APO4719798278.mp3"
# Show example
speaker_df_example = speaker_df1[speaker_df1['mp3url'] == target_url]
for index, row in speaker_df_example.iterrows():
    print(row['turnText'][:150])

As you can see, some values  in turnText only contains "Music] [Music] ... [Music] [Music]", which means that this section of the podcast is currently playing a song. We want to filter  these sections out. We start by lowering everything to lowercase letters. This both makes the filtering  step easier, but  also helps training the model later on  as it helps treat words like "The" and "the" as the same token, reducing vocabulary size.

Additionally, some sections of the turnText string contains multiple whitespace in a row, so we also fix that here, by replacing those with a single space.

In [ ]:
speaker_df1 = speaker_df1.copy()

# Convert text to lowercase
speaker_df1['turnText'] = speaker_df1['turnText'].str.lower()

# Replace multiple spaces with a single space
speaker_df1['turnText'] = speaker_df1['turnText'].str.replace(r'\s+', ' ', regex=True)

# Show example
speaker_df_example = speaker_df1[speaker_df1['mp3url'] == target_url]
for index, row in speaker_df_example.iterrows():
    print(row['turnText'][:150])

Now we can filter  out the music-sections.

In [ ]:
# Remove music indicators from the turnText
# Remove specific music indicators and blank audio patterns
unwanted_patterns_strict = [
    r'\[music\]', r'\[music playing\]', r'\[blank_audio\]',
    r'music\]', r'\[music', r'music playing\]', r'\[music playing',
    r'blank_audio\]', r'\[blank_audio', r'\(music\)'
]
for pattern in unwanted_patterns_strict:
    speaker_df1['turnText'] = speaker_df1['turnText'].str.replace(pattern, '', regex=True)

# Remove multiple whitespaces
speaker_df1['turnText'] = speaker_df1['turnText'].str.replace(r'\s+', ' ', regex=True)

# Remove leading/trailing whitespace
speaker_df1['turnText'] = speaker_df1['turnText'].str.strip()

# Show example
speaker_df_example = speaker_df1[speaker_df1['mp3url'] == target_url]
for index, row in speaker_df_example.iterrows():
    print(row['turnText'][:150])

Now we have almost removed all instances of music-section. Unfortunately, sometimes the music string is seperated like:

[  
music  
]  

which means some of our filtering steps are not being processed properly. We fix this by first removing all rows that only has the words "music", "music playing" and "blank_audio" present.


In [ ]:
import re

# Remove rows that still contains the singles words after cleaning
speaker_df1 = speaker_df1[speaker_df1['turnText'] != 'music']
speaker_df1 = speaker_df1[speaker_df1['turnText'] != 'music playing']
speaker_df1 = speaker_df1[speaker_df1['turnText'] != 'blank_audio']

# Show example
speaker_df_example = speaker_df1[speaker_df1['mp3url'] == target_url]
for index, row in speaker_df_example.iterrows():
    print(row['turnText'][:150])

 Now the single instances "music" has been discarded. However, we  notice that the single characters "[" and "]" are still present. Of course we want to remove these characters. But we have to be careful. We want to ensure that we keep instances of "[laughs]",  "[applauds]" and other types of  reactions that might occur, as removing brackets in such cases could confuse the model into believing that these are  words  being said.  So we define a regex function that only removes brackets if they are empty/invalid.

 Furthermore, we ensure that other singular characters such as ".", "_", ":" ect. also gets filtered out by deleting lines that only contains single character.

In [ ]:
# Define the cleaning regex function
def clean_brackets(text):
    # Keep valid [content], remove stray [ or ]
    return re.sub(r'(\[[^\[\]]+\])|[\[\]]', lambda m: m.group(1) or '', text)

speaker_df1['turnText'] = speaker_df1['turnText'].apply(clean_brackets)

# Remove rows that only contain single characters (or less)
speaker_df1 = speaker_df1[speaker_df1['turnText'].str.len() > 1]

# Show example
speaker_df_example = speaker_df1[speaker_df1['mp3url'] == target_url]
for index, row in speaker_df_example.iterrows():
    print(row['turnText'][:150])

Now we have a decently clean turnText that can be used for the model.

Let's try to find other inconsistencies in the turnText by looking at first 100 lines in the first 100 samples:

In [ ]:
for index, row in speaker_df1.iloc[20:60].iterrows():
    print(row['turnText'][:100], '\n')

We seem to almost be there, having a pretty clean turnText. However, we notice that some lines contains ">>" which is not necessary. So we filter those out as well.

In [ ]:
speaker_df1['turnText'] = speaker_df1['turnText'].str.replace('>>', '', regex=False)

for index, row in speaker_df1.iloc[420:460].iterrows():
    print(row['turnText'][:100], '\n')
    #print(row['speaker'], row['turnText'][:100])

As you can see in this example, we have also managed  to preserve instances of "[laughs]" succesfully.

But another thing we notice is lines consisting of "...". Preserving "..." in longer sentences does make sense as it can give cues that the guest/host is  thinking. But it does not  make sense for single rows, so we filter  these out as well.

In [ ]:
# Remove rows consisting only of '...'
speaker_df1 = speaker_df1[speaker_df1['turnText'] != '...']

# Remove rows that only contain single characters (or less)
speaker_df1 = speaker_df1[speaker_df1['turnText'].str.len() > 1]

for index, row in speaker_df1.iloc[420:460].iterrows():
    print(row['speaker'], row['turnText'][:100], '\n')

# END OF TO BE REMOVED 

--------------------

### 2.4 Token-Level and Structural Cleaning

At this stage, we refine the cleaned transcripts to make them suitable for modeling.
This involves removing punctuation and stopwords, merging consecutive turns from the same speaker, and filtering out low-information text segments.

#### 2.4.1 Text Standardization and Punctuation Removal

We define a text-cleaning function that lowercases the text, trims whitespace, and optionally removes punctuation.
This ensures consistency in tokenization and reduces vocabulary fragmentation

In [ ]:
import string

def clean_text(text, remove_punct=True):
    """
    Lowercase, trim, and optionally remove punctuation.
    Handles missing or non-string entries safely.
    """
    if isinstance(text, str):
        if remove_punct:
            text = text.translate(str.maketrans('', '', string.punctuation))
        return text.lower().strip()
    return text

# Apply the cleaning function
speaker_df1['turnText'] = speaker_df1['turnText'].apply(clean_text)

speaker_df1[['speaker','turnText']][:20]


#### 2.4.2 Merge Consecutive Turns by the Same Speaker

Some transcripts split a speaker’s speech into multiple turns even when uninterrupted.
To maintain coherent speaker contributions, we merge consecutive turns by the same speaker into single entries.

In [ ]:
print(f'Number of turns before merging: {len(speaker_df1)}')

# Ensure the speaker field is a string
speaker_df1['speaker'] = speaker_df1['speaker'].apply(
    lambda x: ','.join(x) if isinstance(x, list) else str(x)
)

# Create a group index that increments when the speaker changes
speaker_df1['group'] = (speaker_df1['speaker'] != speaker_df1['speaker'].shift()).cumsum()

# Merge consecutive text segments by the same speaker
merged_df = (
    speaker_df1
    .groupby(['group', 'speaker'], as_index=False)
    .agg({
        'turnText': lambda x: ' '.join(x),      # concatenate text
        'startTime': 'first',                   # earliest start time
        'endTime': 'last',                      # latest end time
        'mp3url': 'first',                      # keep the episode link
        'inferredSpeakerRole': 'first',         # role stays the same for the speaker
        'turnCount': 'first'                    # optional: could take min() or first()
    })
)

# Turn it back as a list
# Turn it back as a list
merged_df['speaker'] = merged_df['speaker'].apply(
    lambda x: [s.strip() for s in x.split(',')] if isinstance(x, str) else x
)

# Drop the temporary grouping column if not needed
speaker_df1 = merged_df.drop(columns=['group'], errors='ignore')

print(f'Number of turns after merging: {len(speaker_df1)}')

speaker_df1[:20]


#### 2.4.3 Stopword Removal

Stopwords (e.g., the, is, and) are common words that typically carry little semantic meaning.

Removing them reduces noise and emphasizes content-bearing terms.

In [ ]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Remove stopwords from each turn
speaker_df1['filtered_text'] = speaker_df1['turnText'].apply(
    lambda x: ' '.join([word for word in x.split() if word.lower() not in stop_words])
)

speaker_df1[:20]

#### 2.4.4 Remove Extremely Short or Low-Information Turns

Short turns (single words such as “yeah”, “okay”, or “right”) often add noise to topic or embedding models.
We remove these to focus the analysis on meaningful utterances.

In [ ]:
# Keep only turns with more than one word
old_len = len(speaker_df1)
speaker_df1 = speaker_df1[speaker_df1['turnText'].apply(lambda x: len(x.split()) > 1)]

old_len, len(speaker_df1)

In [ ]:
print(len(speaker_df1["turnText"].apply(lambda x: len(x.split()) == 1)))

#### 2.4.5 Example Output

In [ ]:
for index, row in speaker_df1.iloc[420:425].iterrows():
    print(f"Speaker: {row['speaker']}")
    print("Original :", row['turnText'][:100])
    print("Filtered :", row['filtered_text'][:100])
    print()

### 2.5 Episode-Level Preprocessing

#### 2.5.1 Select Relevant Columns
We extract a subset of columns from the original `episodeLevelData` DataFrame, focusing on those that provide contextual and categorical information about each episode.

| Column                           | Description                                                   |
| :------------------------------- | :------------------------------------------------------------ |
| **epTitle**, **epDescription**   | Episode title and description                                 |
| **mp3url**                       | Unique episode identifier (used to link with speaker turns)   |
| **podTitle**, **podDescription** | Podcast-level metadata for context and categorization         |
| **explicit**                     | Flag indicating whether the episode contains explicit content |
| **category1–category10**         | Topical categories associated with the episode                |



In [ ]:
episode_df1 = episode_df[[
    'epTitle', 'mp3url', 'podTitle', 'explicit',
    'category1', 'category2', 'category3', 'category4', 'category5',
    'category6', 'category7', 'category8', 'category9', 'category10'
]].copy()

episode_df1.head()


#### 2.5.2 Handle Missing and Sparse Fields

We first check for missing values in key columns.

Given the dataset’s large size, a small number of missing entries can be safely dropped without compromising representativeness

In [ ]:
print("\nMissing values in episode_df1:")
print(episode_df1[['epTitle', 'mp3url', 'podTitle', 'category1']].isnull().sum())

category_columns = [f'category{i}' for i in range(1, 11)]
unique_episodes_count = len(episode_df1['mp3url'].unique())

print("\nNumber of episodes with each category:")
for col in category_columns:
    count = episode_df1[col].notnull().sum()
    print(f"{col}: {count}/{unique_episodes_count}")


Most episodes include only a few categories, with all values in `category9` and `category10` being empty.

We therefore remove these two columns to reduce sparsity and simplify the dataset.

In [ ]:
print(f"Shape of episode_df1 before dropping missing values and columns: {episode_df1.shape}")

def drop_missing_values_and_columns_episodelvl(episode_dataframe):
    """
    Drop rows missing key metadata and remove unused category columns.
    """
    # Drop rows missing critical fields
    episode_dataframe1 = episode_dataframe.dropna(subset=['podTitle', 'category1'])

    # Drop category9 and category10 if they exist
    columns_to_drop = ['category9', 'category10']
    existing_columns_to_drop = [c for c in columns_to_drop if c in episode_dataframe1.columns]
    if existing_columns_to_drop:
        episode_dataframe1 = episode_dataframe1.drop(columns=existing_columns_to_drop)
    return episode_dataframe1

# Apply the cleanup
episode_df1 = drop_missing_values_and_columns_episodelvl(episode_df1)

print(f"Shape of episode_df1 after dropping missing values and columns: {episode_df1.shape}")


### 2.6 Merge TurnText - and Episode-Level Data

At this stage, we combine the cleaned speaker-turn and episode-level data to form a unified dataset at the episode level.

This step allows us to analyze or model episodes holistically — using both metadata and spoken content.

#### 2.6.1 Aggregate Speaker Turns per Episode

We first group all turnText entries by mp3url (the episode identifier) and concatenate them into a single text block for each episode.

In [ ]:
# Group speaker turns by episode (mp3url) and concatenate the turnText
episode_text_df = (
    speaker_df1
    .groupby('mp3url')['turnText']
    .apply(' '.join)
    .reset_index()
)
episode_text_df.rename(columns={'turnText': 'combined_turnText'}, inplace=True)


#### 2.6.2 Merge with Episode Metadata

We then merge the aggregated text with `episode_df1` using `mp3url` as the join key.

In [ ]:
# Merge the combined text with episode-level dataframe
episode_df1 = pd.merge(episode_df1, episode_text_df, on='mp3url', how='left')

# Inspect a sample row
display(episode_df1.iloc[[2]])


This results in one row per episode, containing:

* metadata (e.g., title, categories),
* and the entire spoken transcript as combined_turnText.

#### 2.6.3 Extract and Summarize Categories

Finally, we extract the unique set of categories across all category columns to understand the topical diversity of the dataset.

In [ ]:
# Extract all category columns
category_columns = [f'category{i}' for i in range(1, 9)]

# Flatten and get unique non-null category values
all_categories = (
    pd.Series(episode_df1[category_columns].values.flatten())
    .dropna()
    .unique()
    .tolist()
)

print(f"Total number of unique categories found: {len(all_categories)}")
print("All unique categories:", all_categories)


### 2.6.4 Examples 

In [ ]:
for i in range(10, 15):
    ep_title = episode_df_train['epTitle'].astype(str).tolist()[i]
    mp3_url = episode_df_train['mp3url'].astype(str).tolist()[i]
    turn_texts = df_train[df_train['mp3url'] == mp3_url]['turnText'].astype(str).tolist()
    print(f"\n{'='*60}")
    print(f"Episode {i+1}")
    print(f"Title      : {ep_title}")
    print(f"MP3 URL    : {mp3_url}")
    print(f"N of turns : {len(turn_texts)}")
    print(f"{'-'*60}")
    print(f"Speaker Turns:")
    speakers = df_train[df_train['mp3url'] == mp3_url]['speaker'].astype(str).tolist()
    for spk, text in zip(speakers, turn_texts):
        print(f"  [{spk}] {text}")
    print(f"{'='*60}\n")

## 3. Split Datasets

In this section of the code we are splitting the datasets into training and validation sets. First we identify the unique podcast episodes using their MP3 URLs. Then divides these unique URLs into an 80% training set and a 20% validation set. Finally, we use these lists of URLs to filter both the speaker turn data and the episode level data, creating corresponding training and validation dataframes for each. This ensures that data from the same episode doesn't end up in both the training and validation sets.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Get unique URLs
uniqueURL = list(set(speaker_df1['mp3url']))

# 2. Split URLs
train_urls, val_urls = train_test_split(uniqueURL, test_size=0.2, random_state=42)

# 3. Filter dataframes
df_train = speaker_df1[speaker_df1['mp3url'].isin(train_urls)]
df_val = speaker_df1[speaker_df1['mp3url'].isin(val_urls)]
episode_df_train = episode_df1[episode_df1['mp3url'].isin(train_urls)]
episode_df_val = episode_df1[episode_df1['mp3url'].isin(val_urls)]

print("Train/Validation Split Overview")
print("-" * 40)
print(f"Total unique URLs: {len(uniqueURL)}")
print(f"Train URLs: {len(train_urls)}")
print(f"Validation URLs: {len(val_urls)}\n")

print(f"Speaker train rows: {len(df_train)}")
print(f"Speaker validation rows: {len(df_val)}")
print(f"Episode train rows: {len(episode_df_train)}")
print(f"Episode validation rows: {len(episode_df_val)}\n")


## 4. Descriptive statistics
### Speaker-Turn Dataset Statistics

This section presents key descriptive statistics for the speaker turn dataset. There are a few key reasons behind why we present these stats:

- **Model Design:** Knowing the typical length of turns helps in choosing appropriate NLP models and setting parameters (eg, maximum sequence length for transformers).
- **Computational Planning:** The number of turns directly impacts the computational resources required for processing.
- **Data Quality:** Identifying missing or malformed text entries is essential for data cleaning and ensuring realible analysis later.
- **Understanding Conversation Structure:** Statistics on turns and speakers per episode provide insights into the nature and structure of the podcast conversations.

By examining these statistics, we gain an understanding of the speaker turn data before proceeding with more complex analyses.

Questions asked:
* How many total turns are there?
* How long are they (average text length)?
* How many turns per episode?
* How many speakers per episode?
* Any missing or malformed text entries?

In [ ]:
import ast
import re

# How many total turns are there?
total_turns = len(speaker_df1)
print(f"Total number of turns: {total_turns}")

# How long are they (average text length)?
average_text_length = speaker_df1['turnText'].str.len().mean()
print(f"Average turn text length: {average_text_length:.2f}")

# How many turns per episode?
turns_per_episode = speaker_df1.groupby('mp3url').size().mean()
print(f"Average turns per episode: {turns_per_episode:.2f}")

# How many speakers per episode?
# The 'speaker' column appears to be lists of speaker labels.
def count_unique_speakers_list(speaker_list):
    # Ensure the input is a list before counting unique speakers
    if isinstance(speaker_list, list):
        return len(set(speaker_list))
    else:
        # Print the problematic entry and its type if it's not a list
        print(f"Unexpected type in speaker column: {speaker_list}, Type: {type(speaker_list)}")
        return 0 # Handle cases where the entry is not a list

speaker_df1['num_speakers'] = speaker_df1['speaker'].apply(count_unique_speakers_list)
speakers_per_episode = speaker_df1.groupby('mp3url')['num_speakers'].max().mean()
print(f"Average speakers per episode: {speakers_per_episode:.2f}")


# Any missing or malformed text entries?
missing_text_entries = speaker_df1['turnText'].isnull().sum()
print(f"Number of missing turnText entries: {missing_text_entries}")

# Check for empty strings in turnText after cleaning
empty_text_entries = (speaker_df1['turnText'] == '').sum()
print(f"Number of empty turnText entries after cleaning: {empty_text_entries}")

Above is shown the answers to the questions asked. Interesting to note is that there 0 missing and empty turnText entries after cleaning, showing that our preprocessing pipeline works as intended.

To get a further/deeper understanding of these numbers, we can make some visualizations showing the distribution of **turnText lengths** and **Turns per episode**.

### Distribution of Turn Text Lengths Visualized

This histogram shows the distribution of the length of each speaker turn in characters. A logarithmic scale is used on the x-axis to accommodate the wide range of turn lengths, including some very long turns. Understanding turn length distribution is important for selecting appropriate NLP models and processing strategies.

In [ ]:
import matplotlib.pyplot as plt

# Plot a histogram of the turn text lengths
plt.figure(figsize=(10, 6))
plt.hist(speaker_df1['turnText'].str.len(), bins=100, edgecolor='black')
plt.title('Distribution of Turn Text Lengths (Logarithmic Scale)')
plt.xlabel('Turn Text Length (Log Scale)')
plt.ylabel('Frequency')
plt.yscale('log')
plt.grid(axis='y', alpha=0.75)
plt.show()

*Note: The y-axis is logarithmic*

An interesting observation here is that the **vast** majority of turnText lengths are very short. This might be indicating that episodes where  multiple speakers are involved, they are often interrupting eachother thus leading to very short sentences/words. This is defintely something to consider  when we input the emotion-model with these short turntexts as this obscure the final distribution of  emotion classification for said episode. So we need to be careful here and consider  solutions...

In [ ]:
import matplotlib.pyplot as plt

# Plot a histogram of the turn text lengths
plt.figure(figsize=(10, 6))
plt.hist(speaker_df1['filtered_text'].str.len(), bins=100, edgecolor='black')
plt.title('Distribution of Turn Text Lengths after cleanup (Logarithmic Scale)')
plt.xlabel('Turn Text Length (Log Scale)')
plt.ylabel('Frequency')
plt.yscale('log')
plt.grid(axis='y', alpha=0.75)
plt.show()

### Distribution of Turns per Episode Visualized
This histogram shows the distribution of the number of speaker turns within each podcast episode. This helps us understand the typical conversational structure and density of the episodes in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Calculate the number of turns per episode
turns_per_episode_counts = speaker_df1.groupby('mp3url').size()

# Plot a histogram of the number of turns per episode
plt.figure(figsize=(10, 6))
plt.hist(turns_per_episode_counts, bins=50, edgecolor='black')
plt.title('Distribution of Turns per Episode')
plt.xlabel('Number of Turns')
plt.ylabel('Number of Episodes')
plt.xlim(0, 2400)
plt.grid(axis='y', alpha=0.75)
plt.show()

An interesting observation here is that most episodes only include a few  amount of turns. This  indicates that many podcasts only has single speakers, thus resulting in one long continuing  turn for the given episode.

## Episode-Level Dataset Statistics

  To ensure that our episode-level analyses and classification tasks are well-grounded, we first need to understand the overall structure and composition of the episode metadata.
  Specifically, we want to examine how categories are distributed across episodes, how many unique topics are represented, and whether the data is complete and consistent enough to support downstream modeling.
  These statistics help us assess the feasibility of episode-level classification and inform key preprocessing decisions such as which category fields to include or discard.






> Before we  look at some statistics for the Episode-Level Dataset, there are  a few things we need to explain. First, each episode contains a "list" of categories expanding from category1 to category10. The main category for a given episode is found in category1  and each subsequent category is other related or secondary categories  for example: epTitle: "Sing Out Speak Out", category1: "music", category2: "comedy" ect. Antoher thing to note is that not all categories is expected  to be  represented for any given episode, for example continouing from "Sing Out Speak Out", category3: None, category4: None, ect.

With that being said, here are the things we wish to answer in following code cell:

- How many unique episodes?

- Total number of unique categories?

- How many episodes contain multiple categories?

- What categories are represented (for each category_x)?

- Are there any missing values in key columns?

In [ ]:
# How many unique episodes?
unique_episodes_count = len(episode_df1['mp3url'].unique())
print(f"Number of unique episodes: {unique_episodes_count}")

# How many different categories in total?
category_columns = [f'category{i}' for i in range(1, 9)]
all_categories = episode_df1[category_columns].values.flatten()
unique_categories = pd.Series(all_categories).dropna().unique()
total_unique_categories = len(unique_categories)
print(f"\nTotal number of unique categories: {total_unique_categories}")

# Number of episodes with each category
print("\nNumber of episodes with each category:")
for col in category_columns:
    count = episode_df1[col].notnull().sum()
    print(f"{col}: {count}/{unique_episodes_count}")


# What categories are represented and any imbalance?
# We can look at the distribution of the first category as a starting point
print("\nDistribution of Category1:")
print('Number of unique categories: ', len(episode_df1['category1'].value_counts()))
print(episode_df1['category1'].value_counts())

print("\nDistribution of Category2:")
print('Number of unique categories: ', len(episode_df1['category2'].value_counts()))
print(episode_df1['category2'].value_counts())

print("\nDistribution of Category3:")
print('Number of unique categories: ', len(episode_df1['category3'].value_counts()))
print(episode_df1['category3'].value_counts())


# Check for missing values in key columns
print("\nMissing values in episode_df1:")
print(episode_df1[['epTitle', 'mp3url', 'podTitle', 'category1']].isnull().sum())


The first thing we find interesting here is that category1 only contains 20 unique categories, while category2 and category3 contain 75 and 74 categories respectively.
We might have expected the main category (category1) to include the broadest variety, since it represents the primary classification of each episode.
However, the opposite pattern suggests that the dataset’s taxonomy is hierarchical: the primary category field (category1) represents a smaller, curated set of high-level genres (e.g., Business, Religion, Sports), while the secondary and tertiary category fields (category2, category3) include more granular or cross-cutting themes (e.g., Football, How to, or Medicine).
This indicates that episodes are first assigned to a broad genre, and then optionally tagged with additional, more specific topics.

Another notable pattern is that the vast majority of episodes are assigned exactly two categories, after which the count of episodes decreases roughly by half with each additional category (from around 8,000 with two categories, to 4,000 with three, and so on).   

Based on these observations, we still need to determine the appropriate level of category granularity for our classification task.
One option is to train the model to predict only the primary category (category1), treating it as a single-label classification problem.
Alternatively, we could extend this to a multi-label setting by including the top 3–5 categories per episode, capturing the broader thematic range that often characterizes podcast content.  
Another evaluation strategy could involve comparing the model’s top-N predicted categories (e.g., top 10) against the ground truth to assess semantic overlap and generalization.
This decision will depend on the trade-off between model complexity, interpretability, and available training data, which we plan to explore further in the next milestone.

### Distribution of Top 20 Most Frequent Categories (all categories)

Here we show the top 20 most frequent categories which are taken from all categories from 1 to 8. This helps us understand the overall thematic landscape of different topics across the dataset, providing insights into the dominant themes and potential areas for more granular analysis.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Combine all category columns into a single Series
category_columns = [f'category{i}' for i in range(1, 9)]
all_categories = episode_df1[category_columns].values.flatten()
all_categories_series = pd.Series(all_categories).dropna()

# Get the top 30 most frequent categories
top_30_categories = all_categories_series.value_counts().head(30)

# Create a bar chart of the top 30 categories
plt.figure(figsize=(10, 6))
top_30_categories.plot(kind='bar')
plt.title('Top 30 Most Frequent Categories (from all categories)')
plt.xlabel('Category')
plt.ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Distribution of Top 20 Most Frequent Categories (from category1)

Here we show the top 20 most frequent categories in category1. This provides insights into the primary genres and main topics covered by the podcasts in the dataset. Analyzing the distribution of the primary category is crucial for understanding the core thematic composition of the data and for informing decisions about episode-level classification tasks.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Combine all category columns into a single Series
all_categories = episode_df1['category1'].values.flatten()
all_categories_series = pd.Series(all_categories).dropna()

# Get the top 20 most frequent categories
top_20_categories = all_categories_series.value_counts().head(20)

# Create a bar chart of the top 20 categories
plt.figure(figsize=(10, 6))
top_20_categories.plot(kind='bar')
plt.title('Top 20 Most Frequent Categories (from category1)')
plt.xlabel('Category')
plt.ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Distribution of Top 30 Most Frequent Categories (from category2)

And here we show the top 30 most frequent categories in category2. Examining this distribution helps reveal common secondary themes or sub-genres present in the podcast episodes.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Combine all category columns into a single Series
all_categories = episode_df1['category2'].values.flatten()
all_categories_series = pd.Series(all_categories).dropna()

# Get the top 30 most frequent categories
top_30_categories = all_categories_series.value_counts().head(30)

# Create a bar chart of the top 30 categories
plt.figure(figsize=(10, 6))
top_30_categories.plot(kind='bar')
plt.title('Top 30 Most Frequent Categories (from category2)')
plt.xlabel('Category')
plt.ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Next Steps

### 5.1 Emotional Detector  (Early Demonstration)

In [ ]:
from transformers import pipeline

# Assume df_speaker contains columns: ['mp3url', 'speaker', 'turnText']
df_sample = speaker_df1.dropna(subset=['turnText']).sample(30, random_state=41)  # tiny sample

# Load emotion detection model
emotion_classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=None)

# Apply model to each turn
results = []
for i, text in enumerate(df_sample['turnText']):
    preds = emotion_classifier(text[:512])  # truncate long texts
    results.append(preds[0])  # get top prediction

# Add results to dataframe
df_sample['emotion'] = [r[0]['label'] for r in results]
df_sample['emotion_score'] = [r[0]['score'] for r in results]


#### 5.1.1 Aggregate emotions per episode

Again needs some work, but this is the core idea illustrating that we are able to take a (random) turnText for a (random) episode and classify(?) an emotion score.

The idea is then that you would be able to see the overall distribution of emotions for a given episode.

(show episode title  instead of url.)

In [ ]:
# Group by episode (mp3url) - group by episode name instead somehow
emotion_counts = (
    df_sample.groupby(['mp3url', 'emotion'])
    .size()
    .unstack(fill_value=0)
)

emotion_ratios = emotion_counts.div(emotion_counts.sum(axis=1), axis=0)
emotion_ratios.head(10)


### 5.1.2 Quick visualization

There are also other emotions, but taken this small  sample (30 turnTexts), those other emotions were never  classified. And of course neutral is the most frequent emotion, as many of the turnTexts contain small sentences or words such as "Yes", "Three" and "This", thus leading to many turnTexts to be classified as neutral.

In [ ]:
import matplotlib.pyplot as plt

emotion_ratios.mean().sort_values().plot(kind='barh')
plt.title("Average Emotion Distribution (sample of 30 turns)")
plt.xlabel("Proportion")
plt.ylabel("Emotion")
plt.show()


#### 5.1.3 Emotional Graph (TO DO)

In [ ]:
# TO DO

### 5.2 Category Classification (TO DO)

In [ ]:
# TO DO

### 5.3 Brand Safety Demonstration (TO DO)

In [ ]:
# TO DO

### 5.4 Audience Alignment Prediction (TO DO)

In [ ]:
# TO DO

# Personal questions  and Todo


*   README.md including abstract, contribrutions(?), methods and proposed timeline - Oguzhan
*   Appendix - Repo organisation (pdf-file) - Oguzhan

*   Refine and further explain PoC section (display episode title instead of mp3url!) - Gustav

*   Include PoC demonstration for category classification? (Just notes) - Gustav

* Add stop words to preprocessing - Cristina

* Think about the emotional graph of an episode - Cristina


how many emotions pr turntext